# 01 · Dataset analysis and preprocessing
**Project:** BharatAssist AI – ML language and safety guard for a multilingual government-scheme assistant
**Dataset:** IndicGuard – multilingual prompt-safety dataset (12 Indian languages, human-labelled)

This notebook covers **Unit 4 (data science with NumPy/pandas)** and uses the package code from
**Units 1–2** (classes, ABC, custom exceptions, file handling, functional programming, multiprocessing).

> **Content note:** the dataset contains harmful prompts. This notebook never prints raw *unsafe*
> prompt text – only counts, charts and safe examples.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

from bharatassist.config import SAMPLE_DIR, FIGURE_DIR, MODEL_DIR, REPORT_DIR

## 1. The raw dataset (full files)
The raw train file is **2.1 GB** – too big to load with `json.load()`. `scripts/sample_data.py`
streams every record with `ijson`, collects statistics, and keeps a random sample per language
(**reservoir sampling**). The three splits are processed **in parallel processes**
(`ProcessPoolExecutor`) with a shared `Lock` guarding the common log file.

In [2]:
with open(SAMPLE_DIR / "raw_stats.json", encoding="utf-8") as fh:
    raw = json.load(fh)

overview = pd.DataFrame({
    split: {
        "Rows": s["rows"],
        "Exact duplicate rows": s["exact_duplicates"],
        "Duplicate %": round(100 * s["exact_duplicates"] / s["rows"], 1),
        "Unique rows": s["rows"] - s["exact_duplicates"],
        "Missing prompt_label": s["missing_prompt_label"],
        "Empty prompt": s["empty_prompt"],
        "Languages": len(s["languages"]),
    } for split, s in raw.items()
}).T
overview.style.format("{:,.0f}").format("{:.1f}", subset=["Duplicate %"])

,Rows,Exact duplicate rows,Duplicate %,Unique rows,Missing prompt_label,Empty prompt,Languages
train,"1,135,176","679,578",59.9,"455,598","21,864","21,888",12
valid,"67,512","39,263",58.2,"28,249","1,032","1,056",12
test,"83,256","45,166",54.2,"38,090",864,864,12


**Finding 1 – massive duplication.** About **60 %** of every split is exact duplicate rows
(same language, prompt and label). Training on duplicates over-weights some prompts and inflates
scores, so they must be removed.

In [3]:
print("Raw spellings of 'tag' (prompt type):")
display(pd.Series(raw["train"]["tags"]).sort_values(ascending=False).to_frame("rows"))
print("Raw spellings of 'language':", sorted(raw["train"]["languages"]))
print("Raw values of 'response_label':", raw["train"]["response_label"])

Raw spellings of 'tag' (prompt type):


,rows
Generic,600140
Jailbreaking,180000
Culture_adaptive,160424
generic,120028
jailbreaking,40000
Jail breaking,20000
Culture Adaptive,14584


Raw spellings of 'language': ['Bengali', 'Gujrati', 'Hindi', 'Kannada', 'Malayalam', 'Marathi', 'Odia', 'Punajbi', 'Tamil', 'Telugu', 'Urdu', 'english']
Raw values of 'response_label': {'None': 446184, "'safe'": 398328, "'unsafe'": 170664, "''": 120000}


**Finding 2 – inconsistent labels.** `tag` has **7 spellings for 3 values**; language names are
misspelt (`Gujrati`, `Punajbi`) or lower-case (`english`); a missing `response_label` appears both
as `null` and as an empty string `''`.

In [4]:
cats = pd.Series(raw["train"]["categories"]).sort_values(ascending=True).tail(12)
from bharatassist.evaluation import bar_h
bar_h(cats / 1000, "Most common harm categories (train, raw)", xlabel="rows (thousands)",
    fmt="{:,.0f}k", path=FIGURE_DIR / "harm_categories.png", figsize=(7.5, 4.6))
plt.show()

**Finding 3 – multi-label column.** `violated_categories` is a comma-separated string with
**23 distinct harm categories**; one prompt can violate several. It needs splitting and multi-hot encoding.

## 2. Load the stratified samples
| Split | Sampled per language | Total |
|---|---|---|
| train | 4,000 | 48,000 |
| valid | 1,000 | 12,000 |
| test  | 1,500 | 18,000 |

The **official splits are kept**: every prompt exists in all 12 languages under the same `id`,
so a random split would leak translations of test prompts into training.

In [5]:
from bharatassist.preprocessing import check_leakage
samples = {s: pd.read_csv(SAMPLE_DIR / f"{s}_sample.csv.gz") for s in ("train", "valid", "test")}
for s, df in samples.items():
    print(f"{s:5s}: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nColumns:", list(samples["train"].columns))
print("Shared prompt ids  train-test:", check_leakage(samples["train"], samples["test"]),
      "| train-valid:", check_leakage(samples["train"], samples["valid"]))
samples["train"].drop(columns=["prompt"]).head()

train: 48,000 rows x 9 columns
valid: 12,000 rows x 9 columns
test : 18,000 rows x 9 columns

Columns: ['split', 'id', 'language', 'prompt', 'prompt_label', 'response_label', 'violated_categories', 'tag', 'prompt_label_source']
Shared prompt ids  train-test: 0 | train-valid: 0


,split,id,language,prompt_label,response_label,violated_categories,tag,prompt_label_source
0,train,b574288bc86246db91ad942be948177e,Bengali,safe,safe,NaN,Generic,human
1,train,9064ee05201c408c98fb59246bdd4813,Bengali,unsafe,NaN,Profanity,Generic,human
2,train,553c64a16ef8435c8e4487137004e2de,Bengali,unsafe,safe,Guns and Illegal Weapons,Generic,human
3,train,914234cfc5c942abb49f4dc18f1297d0,Bengali,safe,safe,NaN,Generic,human
4,train,ed5ad736d0ed4850903a860d4641fcb6,Bengali,unsafe,safe,Criminal Planning/Confessions,Generic,human


In [6]:
print("Missing values per column (train sample):")
samples["train"].isna().sum().to_frame("missing")

Missing values per column (train sample):


,missing
split,0
id,0
language,0
prompt,5
prompt_label,1
response_label,27857
violated_categories,17642
tag,0
prompt_label_source,1


## 3. Preprocessing pipeline
`DataPreprocessor.run()` chains six steps with `functools.reduce` and records how many rows each
step removed:

1. **Normalise** language names, tags and the empty `response_label`
2. **Drop** rows without a `prompt_label` (the target)
3. **Clean text** – Unicode NFC normalisation, remove zero-width characters, replace URLs, collapse whitespace, lower-case Latin script
4. **Drop** prompts that are empty after cleaning
5. **Drop** exact duplicates
6. **Feature engineering** – character length, word count, label encoding (`safe=0`, `unsafe=1`), split categories

In [7]:
from bharatassist.preprocessing import DataPreprocessor, category_matrix

clean, reports = {}, {}
for split, df in samples.items():
    prep = DataPreprocessor()
    clean[split] = prep.run(df)
    reports[split] = prep.report_frame()

reports["train"]

,step,rows_before,rows_after,removed
0,Normalise language / tag / response_label,48000,48000,0
1,Drop rows with missing prompt_label,48000,47999,1
2,"Clean text (NFC, URLs, whitespace, case)",47999,47999,0
3,Drop empty prompts,47999,47995,4
4,Drop exact duplicates,47995,47992,3
5,Add features + encode label,47992,47992,0


In [8]:
print("Tag values before:", samples["train"]["tag"].nunique(), "-> after:", clean["train"]["tag"].nunique())
print("Language values after:", sorted(clean["train"]["language"].unique()))
clean["train"]["tag"].value_counts().to_frame("rows")

Tag values before: 7 -> after: 3
Language values after: ['Bengali', 'English', 'Gujarati', 'Hindi', 'Kannada', 'Malayalam', 'Marathi', 'Odia', 'Punjabi', 'Tamil', 'Telugu', 'Urdu']


,rows
tag,
Generic,28833
Jailbreak,10314
Culture-adaptive,8845


In [9]:
# One safe example, before and after cleaning
safe_en = clean["train"].query("prompt_label == 'safe' and language == 'English'")
changed = safe_en[(safe_en["prompt"].str.len() < 160) & (safe_en["prompt"].str.strip().str.lower() != safe_en["clean_prompt"])]
ex = changed.iloc[0]
print("RAW  :", repr(ex["prompt"][:200]))
print("CLEAN:", repr(ex["clean_prompt"][:200]))

RAW  : 'I meant who is the manager of the Roseville,  CA store?'
CLEAN: 'i meant who is the manager of the roseville, ca store?'


In [10]:
cat_onehot = category_matrix(clean["train"])
print("Multi-hot category matrix:", cat_onehot.shape)
print("Unsafe prompts violating >1 category:",
      f"{(clean['train'].query('label == 1')['n_categories'] > 1).mean():.0%}")
cat_onehot.sum().sort_values(ascending=False).head(8).to_frame("unsafe prompts")

Multi-hot category matrix: (47992, 23)
Unsafe prompts violating >1 category: 56%


,unsafe prompts
Criminal Planning/Confessions,12803
Hate/Identity Hate,5082
Needs Caution,5065
Violence,3978
Harassment,3379
Controlled/Regulated Substances,3241
PII/Privacy,2730
Profanity,2219


## 4. Exploratory analysis of the cleaned training data

In [11]:
from bharatassist.evaluation import label_split_chart, bar_h, BLUE, ORANGE
tr = clean["train"]
label_split_chart(tr["prompt_label"].value_counts(), "Class balance – train (after cleaning)",
                  path=FIGURE_DIR / "class_balance.png")
plt.show()
print(tr["prompt_label"].value_counts(normalize=True).round(3).to_dict())

{'unsafe': 0.573, 'safe': 0.427}


**Finding 4 – class imbalance.** About 57 % unsafe vs 43 % safe. A model that always says
"unsafe" already scores 57 % accuracy, so **accuracy alone is misleading** – we use
**F1 / macro-F1** and train with `class_weight="balanced"`.

In [12]:
lang_counts = tr["language"].value_counts().sort_values()
bar_h(lang_counts, "Training rows per language (after cleaning)", xlabel="rows",
      path=FIGURE_DIR / "rows_per_language.png")
plt.show()

In [13]:
by_lang = (tr.groupby("language")["label"].mean().sort_values() * 100)
bar_h(by_lang, "Share of unsafe prompts per language", xlabel="% unsafe", fmt="{:.0f}%",
      xlim=(0, 100), path=FIGURE_DIR / "unsafe_share_language.png")
plt.show()

In [14]:
stats = tr.groupby("prompt_label")[["char_len", "word_count"]].describe().T
display(stats.round(0))
fig, ax = plt.subplots(figsize=(7, 3.6))
bins = np.linspace(0, 1500, 50)
ax.hist(tr.loc[tr.label == 0, "char_len"].clip(upper=1500), bins=bins, alpha=0.75, color=BLUE, label="safe")
ax.hist(tr.loc[tr.label == 1, "char_len"].clip(upper=1500), bins=bins, alpha=0.6, color=ORANGE, label="unsafe")
ax.set_title("Prompt length in characters", loc="left"); ax.set_xlabel("characters (clipped at 1,500)")
ax.legend(frameon=False); ax.tick_params(length=0)
fig.savefig(FIGURE_DIR / "prompt_length.png", facecolor="white", bbox_inches="tight"); plt.show()

prompt_label         safe   unsafe
char_len   count  20483.0  27509.0
           mean     383.0    293.0
           std      755.0    603.0
           min        1.0      1.0
           25%       50.0     63.0
           50%      127.0    108.0
           75%      424.0    288.0
           max    17652.0  11449.0
word_count count  20483.0  27509.0
           mean      59.0     45.0
           std      116.0     95.0
           min        1.0      1.0
           25%        8.0     10.0
           50%       20.0     17.0
           75%       64.0     42.0
           max     3035.0   2082.0

In [15]:
tag_tab = pd.crosstab(tr["tag"], tr["prompt_label"], normalize="index").round(3) * 100
tag_tab["rows"] = tr["tag"].value_counts()
tag_tab

prompt_label,safe,unsafe,rows
tag,,,
Culture-adaptive,28.0,72.0,8845
Generic,48.9,51.1,28833
Jailbreak,38.0,62.0,10314


**Finding 5 – prompt type matters.** Culture-adaptive prompts are unsafe most often (~72 %),
jailbreak prompts ~62 %, while generic prompts are an even 50/50 mix – the hardest group for a model.

## 5. Save the cleaned data

In [16]:
out_dir = SAMPLE_DIR.parent / "clean"
out_dir.mkdir(exist_ok=True)
cols = ["split", "id", "language", "tag", "clean_prompt", "label", "char_len", "word_count", "n_categories"]
for split, df in clean.items():
    df[cols].to_csv(out_dir / f"{split}_clean.csv.gz", index=False, compression="gzip")
    print(f"{split:5s}: {len(df):,} rows -> {out_dir.name}/{split}_clean.csv.gz")

train: 47,992 rows -> clean/train_clean.csv.gz


valid: 11,986 rows -> clean/valid_clean.csv.gz


test : 17,996 rows -> clean/test_clean.csv.gz


## Summary
| Problem found | Fix |
|---|---|
| ~60 % exact duplicate rows | Deduplicated while streaming and again after cleaning |
| 7 spellings of 3 tags, misspelt language names | Mapping dictionaries (`config.py`) |
| Missing labels; `null` vs `''` | Dropped rows without target; unified missing marker |
| Mixed scripts, invisible characters, URLs | Unicode NFC + regex cleaning |
| Multi-label category string | Split + multi-hot encoding |
| Class imbalance (57/43) | Macro-F1 metric + balanced class weights |
| Translations share an `id` | Official splits kept; leakage check = 0 shared ids |